In [3]:
%run code/losses.py

In [4]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
import torch.nn.functional as F
from transformers import TrainingArguments
from trl import SFTTrainer

from datasets import load_from_disk
# from losses import compute_fkl, compute_rkl 

max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

max_new_tokens = 128
temperature = 2.0

class AdaptiveOPDTrainer(SFTTrainer):
    def __init__(self, *args, teacher_model=None, max_new_tokens=128, temp=2.0, 
                 k_std=1.5, patience=3, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.max_new_tokens = max_new_tokens
        self.temp = temp
        self.tokenizer = kwargs.get("processing_class", kwargs.get("tokenizer"))
        
        # [修改点] 移除固定的首尾阈值，替换为相对阈值的标准差倍数 (k)
        self.k_std = k_std
        self.patience = patience

    def _generate_on_policy(self, model, input_ids, attention_mask):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True,
            )
        if was_training:
            model.train()
        return generated_ids

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompt_input_ids = inputs["input_ids"]
        if "attention_mask" in inputs:
            prompt_attention_mask = inputs["attention_mask"]
        else:
            prompt_attention_mask = prompt_input_ids.ne(self.tokenizer.pad_token_id).long()
            
        prompt_lengths = prompt_attention_mask.sum(dim=1)

        generated_ids = self._generate_on_policy(model, prompt_input_ids, prompt_attention_mask)
        generated_attention_mask = generated_ids.ne(self.tokenizer.pad_token_id).long()

        labels = generated_ids.clone()
        for row_idx, prompt_len in enumerate(prompt_lengths):
            labels[row_idx, :prompt_len] = -100
        labels = labels.masked_fill(generated_attention_mask.eq(0), -100)

        outputs_student = model(
            input_ids=generated_ids,
            attention_mask=generated_attention_mask,
            return_dict=True
        )
        logits = outputs_student.logits

        # =================================================================
        # 基于相对动态阈值与卷积滑窗的截断机制 (Batch-Relative Threshold)
        with torch.no_grad():
            # 1. 计算每一步的预测概率与策略熵 H(p_theta)
            probs = F.softmax(logits, dim=-1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1) # Shape: [batch, seq_len]
            
            # 2. 提取当前 Batch 中所有实际【生成】的 Token 熵
            gen_entropies_list = []
            for b_idx in range(labels.size(0)):
                prompt_len = prompt_lengths[b_idx]
                gen_entropies_list.append(entropy[b_idx, prompt_len:])
            
            all_gen_entropies = torch.cat(gen_entropies_list)
            
            # 3. 计算当前 Batch 所有生成 Token 的平均熵和标准差
            batch_mean = all_gen_entropies.mean()
            batch_std = all_gen_entropies.std()

            # 4. 动态阈值 = 均值 + k倍标准差 (自动适应任何任务难度)
            dynamic_threshold = batch_mean + self.k_std * batch_std
            
            # 提前在 GPU 上构建全 1 卷积核，用于后续的高效滑窗探测
            kernel = torch.ones(1, 1, self.patience, device=logits.device)
            
            # 5. 执行滑窗截断
            for b_idx in range(labels.size(0)):
                prompt_len = prompt_lengths[b_idx]
                gen_entropy = entropy[b_idx, prompt_len:] 
                seq_len = gen_entropy.size(0)
                
                if seq_len < self.patience:
                    continue
                
                # 布尔掩码：0 代表安全，1 代表高熵（触发异常）
                high_entropy_mask = (gen_entropy > dynamic_threshold).float()
                mask_input = high_entropy_mask.unsqueeze(0).unsqueeze(0)
                
                # 滑动窗口计算
                conv_out = F.conv1d(mask_input, kernel).view(-1)
                
                trigger_indices = (conv_out == self.patience).nonzero(as_tuple=True)[0]
                
                if len(trigger_indices) > 0:
                    cutoff_offset = trigger_indices[0].item()
                    actual_cutoff_idx = prompt_len + cutoff_offset
                    # 截断标签
                    labels[b_idx, actual_cutoff_idx:] = -100
        # =================================================================

        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=generated_ids,
                attention_mask=generated_attention_mask,
                return_dict=True
            )
            teacher_logits = teacher_outputs.logits

        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]

            kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=self.temp)
            valid_tokens = labels.ne(-100).sum().clamp_min(1)
            kl = kl / valid_tokens

        loss_total = kl
        return (loss_total, outputs_student) if return_outputs else loss_total

# ------------------------------------------------------------------
# 初始化模型与配置
print("1. 初始化全新的 Student 和 Teacher 模型...")
student, _ = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

student = FastLanguageModel.get_peft_model(
    student,
    r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0, 
    bias="none",    
    use_gradient_checkpointing="unsloth", 
    random_state=3407,
)

teacher, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen_teacher_finetune",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(teacher)
teacher.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dataset = load_from_disk("./data_splits/data_full")
print(f"\n 成功加载全量数据集，数据量: {len(dataset)} 条")

args = TrainingArguments(
    output_dir='./results_adaptive_opd_rel',
    num_train_epochs=4, 
    do_train=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    logging_steps=10,
    save_strategy='no', 
    bf16=True,
    learning_rate=0.0005,
    lr_scheduler_type='constant',
    optim="adamw_torch_fused",
    remove_unused_columns=False,
)

# 传入新参数启动训练
trainer = AdaptiveOPDTrainer(
    model=student,
    teacher_model=teacher,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=args,
    max_new_tokens=max_new_tokens,
    temp=temperature,
    k_std=1.5,           # 动态阈值 = 均值 + 1.5倍标准差
    patience=8           # 放宽耐心值，允许 8 个 Token 的纠错缓冲带
)

import warnings
import transformers

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

print("\n 开始动态截断自适应蒸馏 (Adaptive Truncation OPD - Relative Threshold)")
trainer.train(resume_from_checkpoint=False)

print("\n 训练完成，正在保存模型...")
student.save_pretrained("qwen_student_adaptive_opd_rel")
tokenizer.save_pretrained("qwen_student_adaptive_opd_rel")
print(" 模型已保存至 qwen_student_adaptive_opd_rel 文件夹！")


# qwen_student_adaptive_opd_rel

1. 初始化全新的 Student 和 Teacher 模型...
==((====))==  Unsloth 2026.8.16: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

==((====))==  Unsloth 2026.8.16: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


 成功加载全量数据集，数据量: 2000 条


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]


 开始动态截断自适应蒸馏 (Adaptive Truncation OPD - Relative Threshold)


Step,Training Loss
10,5.095827
20,3.824811
30,3.645390
40,3.579854
50,3.465934
60,3.420501
70,3.243177
80,3.299269
90,3.356834
100,3.322333



 训练完成，正在保存模型...
 模型已保存至 qwen_student_adaptive_opd_rel 文件夹！


In [5]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer


# from losses import compute_fkl, compute_rkl
from datasets import load_from_disk



max_seq_length = 2048
load_in_4bit = True
max_new_tokens = 128
temperature = 2.0


class OPDTrainer(SFTTrainer):
    def __init__(self, *args, teacher_model=None, max_new_tokens=128, temp=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.max_new_tokens = max_new_tokens
        self.temp = temp
        self.tokenizer = kwargs.get("processing_class", kwargs.get("tokenizer"))

    def _generate_on_policy(self, model, input_ids, attention_mask):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True,
            )
        if was_training:
            model.train()
        return generated_ids

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompt_input_ids = inputs["input_ids"]
        prompt_attention_mask = inputs.get("attention_mask", prompt_input_ids.ne(self.tokenizer.pad_token_id).long())
        prompt_lengths = prompt_attention_mask.sum(dim=1)

        generated_ids = self._generate_on_policy(model, prompt_input_ids, prompt_attention_mask)
        generated_attention_mask = generated_ids.ne(self.tokenizer.pad_token_id).long()

        labels = generated_ids.clone()
        for row_idx, prompt_len in enumerate(prompt_lengths):
            labels[row_idx, :prompt_len] = -100
        labels = labels.masked_fill(generated_attention_mask.eq(0), -100)

        outputs_student = model(
            input_ids=generated_ids,
            attention_mask=generated_attention_mask,
            return_dict=True
        )
        logits = outputs_student.logits

        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=generated_ids,
                attention_mask=generated_attention_mask,
                return_dict=True
            )
            teacher_logits = teacher_outputs.logits

        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
            kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=self.temp)
            valid_tokens = labels.ne(-100).sum().clamp_min(1)
            kl = kl / valid_tokens

        loss_total = kl
        return (loss_total, outputs_student) if return_outputs else loss_total


# ---------------------------------------------------------
# 加载模型与数据

print("1. 正在加载【课程学习】训练完成的 Student 模型 (qwen_student_adaptive_opd_rel)...")
student, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen_student_adaptive_opd_rel", #  CL 训练出来的模型
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(student)
student.eval()

print("2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...")
teacher, _ = FastLanguageModel.from_pretrained(
    model_name="qwen_teacher_finetune",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(teacher)
teacher.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("3. 加载测试集...")
test_dataset = load_from_disk("./data_splits/data_test")
print(f" 测试集加载成功，共 {len(test_dataset)} 条数据")

# ---------------------------------------------------------
#  启动评估

args = TrainingArguments(
    output_dir='./eval_results_cl',
    per_device_eval_batch_size=2,
    report_to="none"
)

trainer = OPDTrainer(
    model=student,
    teacher_model=teacher,
    processing_class=tokenizer,
    train_dataset=test_dataset,   # 骗过框架检查
    eval_dataset=test_dataset,    # 真正评估用的是这个
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=args,
    max_new_tokens=max_new_tokens,
    temp=temperature,
)

import warnings
import transformers
# 强行关闭 HuggingFace 各种警告
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

print("\n 开始在测试集上计算 KL 散度...")

# 动态移除导致崩溃的 HTML 进度条，换成纯文本 tqdm
from transformers.trainer_callback import ProgressCallback
callbacks_to_remove = []
for callback in trainer.callback_handler.callbacks:
    if "NotebookProgressCallback" in str(type(callback)):
        callbacks_to_remove.append(callback)
for callback in callbacks_to_remove:
    trainer.remove_callback(callback)
trainer.add_callback(ProgressCallback)

metrics = trainer.evaluate()

print("\n" + "="*50)
print(f" 最终评估结果 (Test Set KL Divergence): {metrics['eval_loss']:.4f}")
print("="*50)

1. 正在加载【课程学习】训练完成的 Student 模型 (qwen_student_adaptive_opd_rel)...
==((====))==  Unsloth 2026.8.16: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...
==((====))==  Unsloth 2026.8.16: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

3. 加载测试集...
 测试集加载成功，共 200 条数据


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/200 [00:00<?, ? examples/s]


 开始在测试集上计算 KL 散度...


  0%|          | 0/100 [00:00<?, ?it/s]

{'eval_loss': '0.2014', 'eval_model_preparation_time': '0.0163', 'eval_runtime': '486.1', 'eval_samples_per_second': '0.411', 'eval_steps_per_second': '0.206', 'epoch': 0}

 最终评估结果 (Test Set KL Divergence): 0.2014
